# Solveur Spectral Signé avec Contraction de Nœuds pour 3-SAT

Ce notebook implémente et évalue un solveur spectral signé pour le problème 3-SAT.

### Caractéristiques et optimisations clés :
1. **Élimination de la partie orientée** : Traduction complète du problème 3-SAT en un graphe quadratique signé (variables, nœud de référence T, variables auxiliaires).
2. **Contraction par Paire Canonique Unique** : Fusion des nœuds auxiliaires uniquement s'ils partagent la même paire orientée canonique. Cela élimine le problème de sur-contrainte transitive qui dégradait les performances spectrale.
3. **Calcul ciblé des valeurs propres ($k=1$)** : 
   * **Sur GPU** : Utilisation de **LOBPCG** (`torch.lobpcg`) itératif très rapide.
   * **Sur CPU** : Utilisation de **SciPy sparse Lanczos** (`scipy.sparse.linalg.eigsh`) en mode shift-invert.
4. **Laplacien Combinatoire par défaut** : Résolution via le Laplacien signé combinatoire (non normalisé) par défaut, avec option pour le Laplacien normalisé.
5. **Recherche Locale WalkSAT optionnelle** : Désactivée par défaut, mais disponible en option pour le raffinement (warm-start).

In [ ]:
# Installation de python-sat si nécessaire
!pip install -q python-sat

In [ ]:
import time
import random
import urllib.request
import numpy as np
import torch
import scipy.sparse
import scipy.sparse.linalg
import matplotlib.pyplot as plt
from pysat.solvers import Glucose4

print("GPU Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

In [ ]:
#@title Configuration de l'Instance { run: "auto" }
source_du_benchmark = "Circuit Miter (Aléatoire)" #@param ["SAT Competition (Industrial)", "Circuit Miter (Aléatoire)", "SATLIB Bounded Model Checking (BMC)", "SATLIB Planning (Blocks World)"]
sat_competition_instance = "Arithmetic Adder (add32)" #@param ["Arithmetic Adder (add32)", "Arithmetic Adder Large (add64)", "Arithmetic Adder Huge (add128)", "Logic Equivalence Miter (miter1)", "Factorization (prime169)", "Factorization Large (prime841)", "Square Root (sqrt10609)", "Square Root Large (sqrt63001)", "XOR Chain (xor6)", "Pigeonhole (ph6)"]
difficulte = "Facile" #@param ["Facile", "Moyen", "Difficile", "Très difficile"]
miter_nombre_portes = 150 #@param {type:"integer"}

COMPETITION_BENCHMARKS = {
    "Arithmetic Adder (add32)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/add32.cnf",
    "Arithmetic Adder Large (add64)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/add64.cnf",
    "Arithmetic Adder Huge (add128)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/add128.cnf",
    "Logic Equivalence Miter (miter1)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/miter1.cnf",
    "Factorization (prime169)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/prime169.cnf",
    "Factorization Large (prime841)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/prime841.cnf",
    "Square Root (sqrt10609)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/sqrt10609.cnf",
    "Square Root Large (sqrt63001)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/sqrt63001.cnf",
    "XOR Chain (xor6)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/xor6.cnf",
    "Pigeonhole (ph6)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/ph6.cnf"
}
PUBLIC_BENCHMARKS = {
    "SATLIB Bounded Model Checking (BMC)": {
        "Facile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/BMC/queueinvar10.cnf",
            "name": "queueinvar10"
        },
        "Moyen": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/BMC/barrel7.cnf",
            "name": "barrel7"
        },
        "Difficile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/BMC/barrel8.cnf",
            "name": "barrel8"
        }
    },
    "SATLIB Planning (Blocks World)": {
        "Facile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/PLANNING/BlocksWorld/bw_large.a.cnf",
            "name": "bw_large.a"
        },
        "Moyen": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/PLANNING/BlocksWorld/bw_large.b.cnf",
            "name": "bw_large.b"
        },
        "Difficile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/PLANNING/BlocksWorld/bw_large.c.cnf",
            "name": "bw_large.c"
        }
    }
}

In [ ]:
def generate_miter_circuit_3sat(n_inputs=30, n_gates=100, diff="Facile"):
    clauses = []
    next_var = n_inputs + 1
    gates1 = []
    gates2 = []
    
    current_vars = list(range(1, n_inputs + 1))
    for _ in range(n_gates):
        gtype = random.choice(["AND", "OR", "XOR"])
        in1 = random.choice(current_vars)
        in2 = random.choice(current_vars)
        out = next_var
        next_var += 1
        gates1.append((gtype, out, in1, in2))
        current_vars.append(out)
    output1 = current_vars[-1]
    
    n_mutations = 1 if diff == "Facile" else (2 if diff == "Moyen" else 4)
    mutated_indices = set(random.sample(range(n_gates), min(n_mutations, n_gates)))
    
    var_offset = next_var - (n_inputs + 1)
    
    for i, (gtype, out, in1, in2) in enumerate(gates1):
        out2 = out + var_offset
        in1_2 = in1 if in1 <= n_inputs else in1 + var_offset
        in2_2 = in2 if in2 <= n_inputs else in2 + var_offset
        
        if i in mutated_indices:
            new_type = "OR" if gtype == "AND" else "AND"
            gates2.append((new_type, out2, in1_2, in2_2))
        else: 
            gates2.append((gtype, out2, in1_2, in2_2))
            
    output2 = current_vars[-1] + var_offset
    next_var = output2 + 1
    
    for g1, g2 in [(gates1, ""), (gates2, "_2")]:
        for gtype, out, in1, in2 in g1:
            if gtype == "AND":
                clauses.append([in1, -out])
                clauses.append([in2, -out])
                clauses.append([-in1, -in2, out])
            elif gtype == "OR":
                clauses.append([-in1, out])
                clauses.append([-in2, out])
                clauses.append([in1, in2, -out])
            elif gtype == "XOR":
                clauses.append([-in1, -in2, -out])
                clauses.append([in1, in2, -out])
                clauses.append([-in1, in2, out])
                clauses.append([in1, -in2, out])
                
    miter_out = next_var
    clauses.append([-output1, -output2, -miter_out])
    clauses.append([output1, output2, -miter_out])
    clauses.append([-output1, output2, miter_out])
    clauses.append([output1, -output2, miter_out])
    clauses.append([miter_out])
    
    return convert_to_3sat(miter_out, clauses)

def parse_dimacs(content):
    clauses = []
    num_vars = 0
    current_clause = []
    for line in content.split('\n'):
        line = line.strip()
        if not line or line.startswith('c'):
            continue
        if line.startswith('p cnf'):
            parts = line.split()
            num_vars = int(parts[2])
            continue
        for token in line.split():
            val = int(token)
            if val == 0:
                if current_clause:
                    clauses.append(current_clause)
                    current_clause = []
            else: 
                current_clause.append(val)
    if current_clause:
        clauses.append(current_clause)
    return num_vars, clauses

def convert_to_3sat(num_vars, clauses):
    final_cnf = []
    next_var = num_vars + 1
    for c in clauses:
        if len(c) <= 3:
            final_cnf.append(c)
        else:
            curr = c
            while len(curr) > 3:
                aux = next_var
                next_var += 1
                final_cnf.append([curr[0], curr[1], aux])
                curr = [-aux] + curr[2:]
            final_cnf.append(curr)
    return next_var - 1, final_cnf

def load_selected_benchmark():
    if source_du_benchmark == "Circuit Miter (Aléatoire)":
        n_inputs = max(5, miter_nombre_portes // 5)
        return generate_miter_circuit_3sat(n_inputs=n_inputs, n_gates=miter_nombre_portes, diff=difficulte)
    elif source_du_benchmark == "SAT Competition (Industrial)":
        url = COMPETITION_BENCHMARKS[sat_competition_instance]
        print(f"\n[Prétraitement] Téléchargement du benchmark : {sat_competition_instance}...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req, timeout=10) as response:
                content = response.read().decode('utf-8')
            num_vars, clauses = parse_dimacs(content)
            return convert_to_3sat(num_vars, clauses)
        except Exception as e:
            print(f"\n[ERREUR] Impossible de télécharger le benchmark ({e}).")
            n_inputs = max(5, miter_nombre_portes // 5)
            return generate_miter_circuit_3sat(n_inputs=n_inputs, n_gates=miter_nombre_portes, diff=difficulte)
    else:
        diff_satlib = difficulte if difficulte in ["Facile", "Moyen", "Difficile"] else "Difficile"
        bench_info = PUBLIC_BENCHMARKS[source_du_benchmark][diff_satlib]
        url = bench_info["url"]
        print(f"\n[Prétraitement] Téléchargement du benchmark : {bench_info['name']}...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req, timeout=10) as response:
                content = response.read().decode('utf-8')
            num_vars, clauses = parse_dimacs(content)
            return convert_to_3sat(num_vars, clauses)
        except Exception as e:
            print(f"\n[ERREUR] Impossible de télécharger le benchmark ({e}).")
            n_inputs = max(5, miter_nombre_portes // 5)
            return generate_miter_circuit_3sat(n_inputs=n_inputs, n_gates=miter_nombre_portes, diff=difficulte)

## 2. Pré-traitement

In [ ]:
def recursive_unit_propagation_and_reductions(num_vars, clauses, active_vars=None, verbose=True):
    if verbose:
        print(f"[Preprocessing] Élimination unitaire récursive & littéraux purs sur {num_vars} variables...")
    
    fixed_literals = {}
    active_clauses = [list(c) for c in clauses]
    if active_vars is None:
        active_vars = set(range(1, num_vars + 1))
    else:
        active_vars = set(active_vars)
    fixed_empty_clauses = 0
    
    changed = True
    while changed:
        changed = False
        
        empty_clauses = sum(1 for c in active_clauses if len(c) == 0)
        if empty_clauses:
            fixed_empty_clauses += empty_clauses
            if verbose:
                print(f"  -> {empty_clauses} clause(s) vide(s) retirée(s).")
            active_clauses = [c for c in active_clauses if len(c) > 0]
        
        unit_counts = {}
        for c in active_clauses:
            if len(c) == 1:
                lit = c[0]
                unit_counts[lit] = unit_counts.get(lit, 0) + 1
                
        if unit_counts:
            var_unit_info = {}
            for lit, count in unit_counts.items():
                var = abs(lit)
                pol = 1 if lit > 0 else -1
                if var not in var_unit_info:
                    var_unit_info[var] = {1: 0, -1: 0}
                var_unit_info[var][pol] = count
                
            for var in sorted(list(var_unit_info.keys())):
                info = var_unit_info[var]
                pos_units = info[1]
                neg_units = info[-1]
                
                if pos_units >= neg_units:
                    l_val = 1
                    m = pos_units
                    opp_units = neg_units
                else:
                    l_val = -1
                    m = neg_units
                    opp_units = pos_units
                    
                opp_clauses_count = 0
                for c in active_clauses:
                    if len(c) >= 2:
                        for lit in c:
                            if abs(lit) == var:
                                pol = 1 if lit > 0 else -1
                                if pol != l_val:
                                    opp_clauses_count += 1
                                    break
                                    
                k = opp_clauses_count + opp_units
                
                if m >= k:
                    if verbose:
                        print(f"  -> Assignation forcée de x_{var} = {l_val} (m={m} >= k={k})")
                    fixed_literals[var] = l_val
                    active_vars.remove(var)
                    
                    new_active_clauses = []
                    for c in active_clauses:
                        satisfied = False
                        for lit in c:
                            if abs(lit) == var:
                                pol = 1 if lit > 0 else -1
                                if pol == l_val:
                                    satisfied = True
                                    break
                        if not satisfied:
                            new_clause = [lit for lit in c if abs(lit) != var]
                            new_active_clauses.append(new_clause)
                    active_clauses = new_active_clauses
                    changed = True
                    break
            if changed:
                continue
                
        pos_counts = {v: 0 for v in active_vars}
        neg_counts = {v: 0 for v in active_vars}
        for c in active_clauses:
            for lit in c:
                var = abs(lit)
                if var in active_vars:
                    if lit > 0:
                        pos_counts[var] += 1
                    else:
                        neg_counts[var] += 1
                        
        pure_vars = []
        for var in list(active_vars):
            pos = pos_counts[var]
            neg = neg_counts[var]
            if pos > 0 and neg == 0:
                pure_vars.append((var, 1))
            elif neg > 0 and pos == 0:
                pure_vars.append((var, -1))
            elif pos == 0 and neg == 0:
                pure_vars.append((var, 1))
                
        if pure_vars:
            changed = True
            for var, val in pure_vars:
                if verbose:
                    print(f"  -> Élimination du littéral pur/orphelin x_{var} = {val}")
                fixed_literals[var] = val
                active_vars.remove(var)
            
            new_active_clauses = []
            for c in active_clauses:
                satisfied = False
                for lit in c:
                    var = abs(lit)
                    val = 1 if lit > 0 else -1
                    if var in fixed_literals and fixed_literals[var] == val:
                        satisfied = True
                        break
                if not satisfied:
                    new_clause = [lit for lit in c if abs(lit) in active_vars]
                    new_active_clauses.append(new_clause)
            active_clauses = new_active_clauses
            
    if verbose:
        print(f"  -> Preprocessing terminé : {len(active_vars)} variables actives, {len(active_clauses)} clauses actives (fixées : {len(fixed_literals)}).")
        if fixed_empty_clauses > 0:
            print(f"  -> Clauses vides cumulées : {fixed_empty_clauses} (contribution constante +{fixed_empty_clauses} * u).")
            
    return active_vars, active_clauses, fixed_literals, fixed_empty_clauses

## 3. Construction du Graphe Signé avec Contraction de Nœuds

In [ ]:
def build_signed_graph_for_3sat(num_vars, active_vars, active_clauses, u=1.0, verbose=True):
    """
    Construit la matrice d'adjacence du graphe signé quadratique pour la formule active.
    En appliquant la contraction explicite (fusion) des variables auxiliaires s_a uniquement 
    lorsqu'elles partagent la même paire orientée canonique. Cela évite le sur-contrainte transitive.
    """
    if verbose:
        print("[Graph Construction] Début de la construction du graphe signé avec contraction canonique...")
        
    var_list = sorted(list(active_vars))
    var_to_idx = {v: i + 1 for i, v in enumerate(var_list)}
    N_red = len(var_list)
    
    # 1. Analyse des clauses ternaires et extraction de la paire canonique
    clause3_list = []
    for c in active_clauses:
        if len(c) == 3:
            v1, v2, v3 = abs(c[0]), abs(c[1]), abs(c[2])
            idx1, idx2, idx3 = var_to_idx[v1], var_to_idx[v2], var_to_idx[v3]
            pol1 = 1 if c[0] > 0 else -1
            pol2 = 1 if c[1] > 0 else -1
            pol3 = 1 if c[2] > 0 else -1
            
            # Tri des littéraux pour définir de façon unique la paire canonique (les 2 premiers)
            lits = [(idx1, pol1), (idx2, pol2), (idx3, pol3)]
            lits.sort(key=lambda x: x[0])
            canonical_pair = (lits[0][0], lits[0][1], lits[1][0], lits[1][1])
            
            clause3_list.append({
                'idx': [idx1, idx2, idx3],
                'pol': [pol1, pol2, pol3],
                'canonical_pair': canonical_pair
            })
            
    # Regroupement des clauses partageant la même paire canonique exacte
    n_clause3 = len(clause3_list)
    adj = {i: [] for i in range(n_clause3)}
    for i in range(n_clause3):
        for j in range(i + 1, n_clause3):
            if clause3_list[i]['canonical_pair'] == clause3_list[j]['canonical_pair']:
                adj[i].append(j)
                adj[j].append(i)
                
    # Composantes connexes de contraction
    visited = [False] * n_clause3
    components = []
    for i in range(n_clause3):
        if not visited[i]:
            comp = []
            queue = [i]
            visited[i] = True
            while queue:
                curr = queue.pop(0)
                comp.append(curr)
                for neighbor in adj[curr]:
                    if not visited[neighbor]:
                        visited[neighbor] = True
                        queue.append(neighbor)
            components.append(comp)
            
    # Affectation des indices de nœuds fusionnés (commencent après N_red)
    clause_to_merged_idx = {}
    n_merged_nodes = len(components)
    for comp_idx, comp in enumerate(components):
        merged_node = N_red + 1 + comp_idx
        for clause_idx in comp:
            clause_to_merged_idx[clause_idx] = merged_node
            
    if verbose:
        print(f"  -> {n_clause3} clauses ternaires regroupées en {n_merged_nodes} variables auxiliaires distinctes (contraction canonique).")
        
    # 2. Construction de la liste des arêtes
    edges = []
    clause3_counter = 0
    for c in active_clauses:
        if len(c) == 1:
            lit = c[0]
            v = abs(lit)
            idx = var_to_idx[v]
            pol = 1 if lit > 0 else -1
            edges.append((idx, 0, u, pol))
            
        elif len(c) == 2:
            v1, v2 = abs(c[0]), abs(c[1])
            idx1, idx2 = var_to_idx[v1], var_to_idx[v2]
            pol1 = 1 if c[0] > 0 else -1
            pol2 = 1 if c[1] > 0 else -1
            edges.append((idx1, 0, u / 2.0, pol1))
            edges.append((idx2, 0, u / 2.0, pol2))
            edges.append((idx1, idx2, u / 2.0, -pol1 * pol2))
            
        elif len(c) == 3:
            info = clause3_list[clause3_counter]
            idx1, idx2, idx3 = info['idx']
            pol1, pol2, pol3 = info['pol']
            s = clause_to_merged_idx[clause3_counter]
            
            # 3 arêtes (v_i, T) de poids u/2 et de signe pol_i
            edges.append((idx1, 0, u / 2.0, pol1))
            edges.append((idx2, 0, u / 2.0, pol2))
            edges.append((idx3, 0, u / 2.0, pol3))
            
            # 3 arêtes (s, v_i) de poids u/2 et de signe pol_i
            edges.append((s, idx1, u / 2.0, pol1))
            edges.append((s, idx2, u / 2.0, pol2))
            edges.append((s, idx3, u / 2.0, pol3))
            
            # 1 arête (s, T) de poids u/2 et de signe -1
            edges.append((s, 0, u / 2.0, -1))
            
            # 3 arêtes (v_i, v_j) de poids u/2 et de signe -pol_i * pol_j
            edges.append((idx1, idx2, u / 2.0, -pol1 * pol2))
            edges.append((idx2, idx3, u / 2.0, -pol2 * pol3))
            edges.append((idx1, idx3, u / 2.0, -pol1 * pol3))
            
            clause3_counter += 1
            
    total_nodes = N_red + 1 + n_merged_nodes
    A = np.zeros((total_nodes, total_nodes), dtype=np.float32)
    for u_node, v_node, weight, sign in edges:
        val = sign * weight
        A[u_node, v_node] += val
        A[v_node, u_node] += val
        
    if verbose:
        print(f"  -> Graphe complet construit : {total_nodes} nœuds (T, {N_red} variables actives, {n_merged_nodes} variables auxiliaires).")
        
    return A, var_to_idx

## 4. Résolution Spectrale Ciblée (k=1)

In [ ]:
def solve_signed_spectral_clustering(A, var_to_idx, active_vars, fixed_literals, clauses, normalize=True, verbose=True):
    """
    Résout le problème de clustering signé en trouvant UNIQUEMENT le premier vecteur propre
    (associé à la plus petite valeur propre) via des solveurs creux itératifs (LOBPCG sur GPU / Lanczos sur CPU).
    """
    N = A.shape[0]
    N_red = len(active_vars)
    
    if verbose:
        print(f"[Spectral Solver] Construction du Laplacien signé ({'Normalisé' if normalize else 'Combinatoire'}) sur {N} nœuds...")
        
    # Formulation creuse du Laplacien L_signed = D - A
    A_sparse = scipy.sparse.csc_matrix(A)
    d = np.array(np.sum(np.abs(A), axis=1)).flatten()
    D_sparse = scipy.sparse.diags(d)
    L_signed = D_sparse - A_sparse
    
    if normalize:
        d_inv_sqrt = np.zeros_like(d)
        mask = d > 1e-9
        d_inv_sqrt[mask] = 1.0 / np.sqrt(d[mask])
        D_inv_sqrt = scipy.sparse.diags(d_inv_sqrt)
        L_to_solve = D_inv_sqrt @ L_signed @ D_inv_sqrt
    else:
        L_to_solve = L_signed
        
    t_start = time.time()
    # Résolution ciblée (uniquement la plus petite valeur propre)
    if torch.cuda.is_available():
        if verbose:
            print("  -> Résolution sur GPU (PyTorch CUDA via LOBPCG)...(k=1, target: smallest)")
        L_dense = L_to_solve.toarray()
        L_cuda = torch.tensor(L_dense, dtype=torch.float32, device='cuda')
        X = torch.randn(N, 1, device='cuda')
        eigenvalues, eigenvectors = torch.lobpcg(L_cuda, k=1, largest=False, X=X)
        v_min = eigenvectors[:, 0].cpu().numpy()
        eigenval = eigenvalues[0].item()
    else:
        if verbose:
            print("  -> Résolution sur CPU (scipy.sparse.linalg.eigsh shift-invert)...(k=1, target: smallest)")
        eigenvalues, eigenvectors = scipy.sparse.linalg.eigsh(L_to_solve, k=1, sigma=-1e-5, which='LM')
        v_min = eigenvectors[:, 0]
        eigenval = eigenvalues[0]
        
    t_solve = time.time() - t_start
    if verbose:
        print(f"  -> Étape de résolution terminée en {t_solve:.4f}s. Plus petite valeur propre = {eigenval:.6f}")
        
    if normalize:
        v_min = D_inv_sqrt @ v_min
        
    # Alignement du vecteur propre par rapport au spin de référence T (nœud 0)
    T_val = v_min[0]
    T_sign = np.sign(T_val) if abs(T_val) > 1e-9 else 1.0
    aligned_v = v_min * T_sign
        
    # Évaluation des énergies (nombre de clauses insatisfaites)
    def evaluate_energy(spins):
        unsat = 0
        for c in clauses:
            sat = False
            for lit in c:
                val = spins.get(abs(lit), 1)
                pol = 1 if lit > 0 else -1
                if val == pol:
                    sat = True
                    break
            if not sat:
                unsat += 1
        return unsat
        
    # Arrondi par balayage (Sweep Rounding) optimisé
    var_list = sorted(list(active_vars))
    aligned_vals = np.array([aligned_v[var_to_idx[v]] for v in var_list])
    thresholds = [0.0]
    if len(aligned_vals) > 1:
        thresholds.extend(np.percentile(aligned_vals, np.linspace(5, 95, 19)))
    thresholds = sorted(list(set(thresholds)))
    
    best_spins = None
    best_unsat = len(clauses) + 1
    
    for theta in thresholds:
        base_spins = np.where(aligned_vals >= theta, 1, -1)
        for sign in [1, -1]:
            cand = dict(zip(var_list, base_spins * sign))
            cand.update(fixed_literals)
            
            unsat = evaluate_energy(cand)
            if unsat < best_unsat:
                best_unsat = unsat
                best_spins = cand
        
    aligned_v_dict = {}
    for var in active_vars:
        node_idx = var_to_idx[var]
        aligned_v_dict[var] = float(aligned_v[node_idx])
        
    return best_spins, best_unsat, t_solve, aligned_v_dict

## 5. Phase d'Hybridation Optionnelle (WalkSAT Local Search)

In [ ]:
def run_walksat(initial_spins, clauses, num_vars, max_flips=200000, p=0.3, max_time=5.0, verbose=True):
    """
    WalkSAT local search solver.
    Démarré avec la solution du clustering spectral (warm-start).
    """
    t_start = time.time()
    spins = np.zeros(num_vars + 1, dtype=int)
    for v in range(1, num_vars + 1):
        if initial_spins is None or v not in initial_spins:
            spins[v] = random.choice([-1, 1])
        else:
            spins[v] = initial_spins[v]
        
    pos_clauses = [[] for _ in range(num_vars + 1)]
    neg_clauses = [[] for _ in range(num_vars + 1)]
    for c_idx, c in enumerate(clauses):
        for lit in c:
            v = abs(lit)
            if lit > 0:
                pos_clauses[v].append(c_idx)
            else:
                neg_clauses[v].append(c_idx)
                
    num_satisfied = np.zeros(len(clauses), dtype=int)
    unsat_list = []
    clause_to_idx = np.full(len(clauses), -1, dtype=int)
    
    for c_idx, c in enumerate(clauses):
        sat_count = sum(1 for lit in c if spins[abs(lit)] == (1 if lit > 0 else -1))
        num_satisfied[c_idx] = sat_count
        if sat_count == 0:
            clause_to_idx[c_idx] = len(unsat_list)
            unsat_list.append(c_idx)
            
    init_unsat = len(unsat_list)
    if init_unsat == 0:
        best_spins_dict = {v: int(spins[v]) for v in range(1, num_vars + 1)}
        return best_spins_dict, 0, 0.0, 0
        
    best_unsat = init_unsat
    best_spins = spins.copy()
    
    for flip in range(1, max_flips + 1):
        if not unsat_list:
            break
        if flip % 4000 == 0 and time.time() - t_start > max_time:
            break
            
        c_idx = random.choice(unsat_list)
        c = clauses[c_idx]
        
        if random.random() < p:
            flip_var = abs(random.choice(c))
        else: 
            best_diff = -1e9
            best_vars = []
            for lit in c:
                v = abs(lit)
                if spins[v] == 1:
                    lose_clauses = pos_clauses[v]
                    gain_clauses = neg_clauses[v]
                else:
                    lose_clauses = neg_clauses[v]
                    gain_clauses = pos_clauses[v]
                    
                broken = sum(1 for tc_idx in lose_clauses if num_satisfied[tc_idx] == 1)
                made = sum(1 for tc_idx in gain_clauses if num_satisfied[tc_idx] == 0)
                
                diff = made - broken
                if diff > best_diff:
                    best_diff = diff
                    best_vars = [v]
                elif diff == best_diff:
                    best_vars.append(v)
                    
            flip_var = random.choice(best_vars)
            
        old_val = spins[flip_var]
        new_val = -old_val
        spins[flip_var] = new_val
        
        if old_val == 1:
            lose_clauses = pos_clauses[flip_var]
            gain_clauses = neg_clauses[flip_var]
        else:
            lose_clauses = neg_clauses[flip_var]
            gain_clauses = pos_clauses[flip_var]
            
        for tc_idx in lose_clauses:
            num_satisfied[tc_idx] -= 1
            if num_satisfied[tc_idx] == 0:
                if clause_to_idx[tc_idx] == -1:
                    clause_to_idx[tc_idx] = len(unsat_list)
                    unsat_list.append(tc_idx)
                    
        for tc_idx in gain_clauses:
            if num_satisfied[tc_idx] == 0:
                idx = clause_to_idx[tc_idx]
                if idx != -1:
                    last_c_idx = unsat_list[-1]
                    unsat_list[idx] = last_c_idx
                    clause_to_idx[last_c_idx] = idx
                    unsat_list.pop()
                    clause_to_idx[tc_idx] = -1
            num_satisfied[tc_idx] += 1
            
        num_unsat = len(unsat_list)
        if num_unsat < best_unsat:
            best_unsat = num_unsat
            best_spins = spins.copy()
            if best_unsat == 0:
                break
                
    t_elapsed = time.time() - t_start
    if verbose:
        print(f"  -> WalkSAT : Unsat final = {best_unsat} | Flips = {flip} | Temps = {t_elapsed:.4f}s")
        
    best_spins_dict = {v: int(best_spins[v]) for v in range(1, num_vars + 1)}
    return best_spins_dict, best_unsat, t_elapsed, flip

## 6. Baseline CDCL (PySAT Glucose4)

In [ ]:
def solve_3sat_with_pysat(num_vars, clauses, verbose=True):
    t_start = time.time()
    if verbose:
        print("[CPU PySAT] Lancement de la baseline Glucose4...")
    with Glucose4(bootstrap_with=clauses) as solver:
        satisfied = solver.solve()
        model = solver.get_model() if satisfied else None
        
    t_elapsed = time.time() - t_start
    if model:
        if verbose:
            print(f"  -> Glucose4 a résolu le problème (SAT) en {t_elapsed:.4f}s.")
        spins = {abs(lit): (1 if lit > 0 else -1) for lit in model}
        return spins, t_elapsed, 0
    else:
        if verbose:
            print(f"  -> Glucose4 a conclu à UNSAT en {t_elapsed:.4f}s.")
        default_spins = {v: 1 for v in range(1, num_vars + 1)}
        def evaluate_energy(spins):
            unsat = 0
            for c in clauses:
                sat = False
                for lit in c:
                    val = spins.get(abs(lit), 1)
                    pol = 1 if lit > 0 else -1
                    if val == pol:
                        sat = True
                        break
                if not sat:
                    unsat += 1
            return unsat
        return default_spins, t_elapsed, evaluate_energy(default_spins)

def solve_maxsat_with_pysat(num_vars, clauses, verbose=True):
    t_start = time.time()
    if verbose:
        print("[CPU PySAT] Lancement de la baseline MaxSAT exacte (RC2)...")
    if len(clauses) > 3000:
        if verbose:
            print("  -> Instance trop grande (>3000 clauses), saut de la résolution MaxSAT exacte.")
        return None, 0.0, -1
    from pysat.examples.rc2 import RC2
    from pysat.formula import WCNF
    wcnf = WCNF()
    for c in clauses:
        wcnf.append(c, weight=1)
    try:
        with RC2(wcnf) as rc2:
            model = rc2.compute()
            t_elapsed = time.time() - t_start
            if model:
                opt_unsat = rc2.cost
                spins = {abs(lit): (1 if lit > 0 else -1) for lit in model}
                if verbose:
                    print(f"  -> RC2 a résolu le problème MaxSAT (Unsat = {opt_unsat}) en {t_elapsed:.4f}s.")
                return spins, t_elapsed, opt_unsat
    except Exception as e:
        if verbose:
            print(f"  -> RC2 a rencontré une erreur : {e}")
    return None, time.time() - t_start, -1

## 7. Orchestration et Évaluation

In [ ]:
def simplify_clauses_with_assignments(clauses, assignments):
    new_clauses = []
    for c in clauses:
        satisfied = False
        for lit in c:
            var = abs(lit)
            if var in assignments:
                val = assignments[var]
                pol = 1 if lit > 0 else -1
                if val == pol:
                    satisfied = True
                    break
        if not satisfied:
            new_clause = [lit for lit in c if abs(lit) not in assignments]
            new_clauses.append(new_clause)
    return new_clauses

def solve_3sat_spectral_decimation(num_vars, clauses, decimation_fraction=0.05, u=1.0, normalize=True, verbose=True):
    """
    Solveur spectral signé avec décimation stochastique déterministe (Option 2).
    """
    t_start = time.time()
    
    current_clauses = [list(c) for c in clauses]
    all_fixed_literals = {}
    step = 0
    active_vars = set(range(1, num_vars + 1))
    
    while True:
        step += 1
        if verbose:
            print(f"\n--- [Décimation Spectrale] Étape {step} ---")
            
        # 1. Preprocessing récursif
        active_vars, active_clauses, fixed_lits, empty_count = recursive_unit_propagation_and_reductions(
            num_vars, current_clauses, active_vars=active_vars, verbose=verbose
        )
        
        for var, val in fixed_lits.items():
            all_fixed_literals[var] = val
        current_clauses = active_clauses
    
        if len(active_vars) < 10:
            if len(active_vars) > 0:
                if verbose:
                    print(f"[Décimation Spectrale] Moins de 10 variables actives ({len(active_vars)}). Résolution finale en une passe spectrale...")
                A, var_to_idx = build_signed_graph_for_3sat(num_vars, active_vars, current_clauses, u=u, verbose=verbose)
                try:
                    spec_spins, _, _, _ = solve_signed_spectral_clustering(
                        A, var_to_idx, active_vars, {}, current_clauses, normalize=normalize, verbose=verbose
                    )
                except Exception as e:
                    if verbose:
                        print(f"  [Décimation Finale] Échec du solveur spectral ({e}). Repli sur une résolution brute-force...")
                    best_unsat = len(current_clauses) + 1
                    best_spins = {}
                    active_vars_list = list(active_vars)
                    for i in range(1 << len(active_vars_list)):
                        candidate = {}
                        for j, var in enumerate(active_vars_list):
                            candidate[var] = 1 if (i & (1 << j)) else -1
                        unsat = 0
                        for c in current_clauses:
                            sat = False
                            for lit in c:
                                val = candidate.get(abs(lit), 1)
                                pol = 1 if lit > 0 else -1
                                if val == pol:
                                    sat = True
                                    break
                            if not sat:
                                unsat += 1
                        if unsat < best_unsat:
                            best_unsat = unsat
                            best_spins = candidate
                    spec_spins = best_spins
                for var in active_vars:
                    all_fixed_literals[var] = spec_spins[var]
                    if verbose:
                        print(f"  [Décimation Finale] Variable x_{var} = {spec_spins[var]}")
            else:
                if verbose:
                    print("[Décimation Spectrale] Toutes les variables ont été assignées.")
            break
            
        # 2. Construction du graphe signé
        A, var_to_idx = build_signed_graph_for_3sat(num_vars, active_vars, current_clauses, u=u, verbose=verbose)
        
        # 3. Clustering Spectral Signé (k=1)
        _, _, _, aligned_v_dict = solve_signed_spectral_clustering(
            A, var_to_idx, active_vars, {}, current_clauses, normalize=normalize, verbose=verbose
        )
        
        # 4. Tri par confiance spectrale (|v_i|)
        sorted_active_vars = sorted(list(active_vars), key=lambda v: abs(aligned_v_dict[v]), reverse=True)
        n_to_fix = max(1, int(np.ceil(decimation_fraction * len(active_vars))))
        vars_to_fix = sorted_active_vars[:n_to_fix]
        
        decimated_assignments = {}
        for var in vars_to_fix:
            val = 1 if aligned_v_dict[var] >= 0 else -1
            decimated_assignments[var] = val
            all_fixed_literals[var] = val
            active_vars.remove(var)
            if verbose:
                print(f"  [Décimation] Fixation de x_{var} = {val} (confiance: {abs(aligned_v_dict[var]):.6f})")
                
        # 5. Simplification de la formule
        current_clauses = simplify_clauses_with_assignments(current_clauses, decimated_assignments)
        
    # Remplir les variables non définies
    for v in range(1, num_vars + 1):
        if v not in all_fixed_literals:
            all_fixed_literals[v] = 1
            
    def evaluate_energy(spins):
        unsat = 0
        for c in clauses:
            sat = False
            for lit in c:
                val = spins.get(abs(lit), 1)
                pol = 1 if lit > 0 else -1
                if val == pol:
                    sat = True
                    break
            if not sat:
                unsat += 1
        return unsat
        
    final_unsat = evaluate_energy(all_fixed_literals)
    t_elapsed = time.time() - t_start
    
    if verbose:
        print(f"[Décimation Spectrale] Terminé en {t_elapsed:.4f}s. Clauses insatisfaites totales : {final_unsat}")
        
    return all_fixed_literals, t_elapsed, final_unsat

def solve_3sat_spectral(num_vars, clauses, u=1.0, normalize=True, run_local_search=False, verbose=True):
    """
    Fonction d'orchestration pour le solveur spectral signé (une seule passe, sans décimation).
    """
    t_start = time.time()
    
    active_vars, active_clauses, fixed_lits, empty_count = recursive_unit_propagation_and_reductions(num_vars, clauses, verbose=verbose)
    
    if len(active_vars) == 0:
        if verbose:
            print("[Orchestration] Toutes les variables ont été résolues lors du pré-traitement !")
        best_spins = {v: fixed_lits.get(v, 1) for v in range(1, num_vars + 1)}
        return best_spins, time.time() - t_start, empty_count
        
    A, var_to_idx = build_signed_graph_for_3sat(num_vars, active_vars, active_clauses, u=u, verbose=verbose)
    
    spec_spins, spec_unsat, t_solve, aligned_v_dict = solve_signed_spectral_clustering(A, var_to_idx, active_vars, fixed_lits, clauses, normalize=normalize, verbose=verbose)
    
    final_spins = spec_spins
    final_unsat = spec_unsat
    
    if run_local_search:
        if verbose:
            print(f"\n[Orchestration] Lancement de la recherche locale WalkSAT (warm-start)...")
        hybrid_spins, walksat_unsat, t_walk, flips_walk = run_walksat(spec_spins, clauses, num_vars, max_flips=300000, verbose=verbose)
        final_spins = hybrid_spins
        final_unsat = walksat_unsat
    
    total_unsat = final_unsat
    t_elapsed = time.time() - t_start
    
    if verbose:
        print(f"[Orchestration] Terminement en {t_elapsed:.4f}s. Clauses insatisfaites totales : {total_unsat} (dont {empty_count} vides de pré-traitement).")
        
    return final_spins, t_elapsed, total_unsat

In [ ]:
verbose = True

num_vars, clauses_3sat = load_selected_benchmark()
print(f"\n--- Évaluation : {num_vars} variables, {len(clauses_3sat)} clauses ---")

# 1. Notre solveur spectral signé par défaut (Normalisé, sans recherche locale, une seule passe)
print("\n[Exécution] Solveur Spectral Signé (Laplacien Normalisé par défaut, une seule passe, sans WalkSAT)...")
spectral_spins, spectral_time, spectral_unsat = solve_3sat_spectral(num_vars, clauses_3sat, u=1.0, normalize=True, run_local_search=False, verbose=verbose)

# 2. Notre solveur spectral signé avec DÉCIMATION stochastique déterministe (Option 2, 5% par étape, Normalisé)
print("\n[Exécution] Solveur Spectral Signé avec DÉCIMATION (Laplacien Normalisé, 5% par étape, sans WalkSAT)...")
decim_spins, decim_time, decim_unsat = solve_3sat_spectral_decimation(num_vars, clauses_3sat, decimation_fraction=0.05, u=1.0, normalize=True, verbose=verbose)

# 3. Chaîne complète : Décimation Spectrale (5%) + WalkSAT
print("\n[Exécution] Chaîne Complète : Décimation Spectrale (5%) + WalkSAT...")
hybrid_spins, hybrid_unsat, t_walk_hybrid, flips_hybrid = run_walksat(decim_spins, clauses_3sat, num_vars, max_flips=300000, verbose=verbose)
total_hybrid_time = decim_time + t_walk_hybrid

# 4. WalkSAT Pur (Random-start) pour comparaison de flips/temps pour atteindre les mêmes résultats
print("\n[Exécution] WalkSAT Pur (Random-start) ciblant les mêmes résultats...")
pure_spins, pure_unsat, t_walk_pure, flips_pure = run_walksat(None, clauses_3sat, num_vars, max_flips=300000, verbose=verbose)

# 5. Baseline CDCL Glucose4
print("\n[Exécution] Baseline CDCL Glucose4...")
cpu_spins, cpu_time, cpu_unsat = solve_3sat_with_pysat(num_vars, clauses_3sat, verbose=verbose)

# 6. Baseline MaxSAT Exacte RC2
print("\n[Exécution] Baseline MaxSAT Exacte RC2...")
maxsat_spins, maxsat_time, maxsat_unsat = solve_maxsat_with_pysat(num_vars, clauses_3sat, verbose=verbose)

# Comparaison finale
print("\n=== BILAN COMPARATIF ===")
print(f"Solveur Spectral (Une passe, Normalisé)       : Unsat = {spectral_unsat} | Temps = {spectral_time:.4f} s")
print(f"Solveur Spectral (Décimation 5%, Normalisé)   : Unsat = {decim_unsat} | Temps = {decim_time:.4f} s")
print(f"Chaîne Complète (Décimation 5% + WalkSAT)     : Unsat = {hybrid_unsat} | Temps Total = {total_hybrid_time:.4f} s (WalkSAT: {t_walk_hybrid:.4f} s, {flips_hybrid} flips)")
print(f"WalkSAT Pur (Random-start, max 300k flips)    : Unsat = {pure_unsat} | Temps = {t_walk_pure:.4f} s ({flips_pure} flips)")
print(f"Baseline CDCL Glucose4                        : Unsat = {cpu_unsat} | Temps = {cpu_time:.4f} s")
print(f"Baseline MaxSAT RC2 (Exacte)                  : Unsat = {maxsat_unsat} | Temps = {maxsat_time:.4f} s")